# Figure 6 — panel preparation

## Setup

In [ ]:
library(data.table)
library(tidyverse)

## 6b: FUCA2 + APOD trans circos

In [ ]:
# === Figure 6b: FUCA2 (trans regulator) + APOD (trans target) circos ===
suppressMessages({library(data.table);library(tidyverse);library(circlize)})
OUT <- 'data'

gene_ref <- fread('resource/references/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list') %>%
  setNames(c('chr_gene','start_gene','end_gene','gene_id','gene_name'))
qtl_trans <- fread('data_output/trans/fine_mapping//ROSMAP_gpQTL.MAF_P_filtered.exported.toploci.bed.gz')
all_trans_95 <- qtl_trans %>% distinct %>% mutate(gpID=str_extract(event_ID,'gp_.+'), p=2*pnorm(-abs(z))) %>%
  filter(cs_coverage_0.95>0) %>% mutate(cs_id=paste(gene_ID,event_ID,cs_coverage_0.95,sep='_'))
id_mapping <- fread('gp_coordinates_clean.txt')   # gpID -> ensembl_id (trans target gene)
flt <- all_trans_95 %>%
  merge(id_mapping %>% select(-c(1:3)), by.x='gpID', by.y='ID') %>%
  merge(gene_ref %>% setNames(c('#chr_gene','start_gene','end_gene','gene_id','gene_name')),
        by.x='ensembl_id', by.y='gene_id') %>%
  filter(MAF>=0.05, p<5e-8)

qc_remove_gpids <- readLines('../../2_main_figures/Figure1/data/qc_remove_gpids.txt')
w_sole_exclude_gpids <- readLines('../../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')
stopifnot(sum(unique(flt$gpID) %in% qc_remove_gpids) == 0, sum(unique(flt$gpID) %in% w_sole_exclude_gpids) == 8)  # sanity: expected pre-exclusion overlap
flt <- flt %>% filter(!gpID %in% c(qc_remove_gpids, w_sole_exclude_gpids))
stopifnot(!any(flt$gene_name %in% c('APOD') & flt$gpID %in% c(qc_remove_gpids, w_sole_exclude_gpids)))

fuca2_cis_vars <- fread(cmd="zcat annotation_files/xQTL_toploci/Fungen_xQTL.Mar2026.xQTL.multigene.multicontext.colocboost.GWAS.export.csv.gz | awk -F',' 'NR>1 && $7==\"ENSG00000001036\" && $9>0 {print $6}' | sort -u", header=FALSE)$V1

chrom_sizes <- data.frame(chr=paste0('chr',1:22), size=c(248956422,242193529,198295559,190214555,181538259,170805979,159345973,145138636,138394717,133797422,135086622,133275309,114364328,107043718,101991189,90338345,83257441,80373285,58617616,64444167,46709983,50818468))
csz <- setNames(chrom_sizes$size, chrom_sizes$chr)
clamp <- function(chr,pos) pmin(pmax(pos,1), csz[chr]-1e5)
trans_blue <- '#4393C3'
c_trans_blue <- '#92C5DE'
highlight_blue <- '#08519C'
cis_red <- '#D6604D'
setup_circos <- function(label_df, outfile, label_cex=1, label_height_cm=1, device_height=8, chr_cex=1.35){
  pdf(outfile,width=10.5,height=device_height); circos.clear()
  circos.par(start.degree=90, gap.degree=1, track.margin=c(0.003,0.003),
             cell.padding=c(0,0,0,0), canvas.xlim=c(-1.55,1.55), canvas.ylim=c(-1.55,1.55),
             points.overflow.warning=FALSE)
  circos.initialize(chrom_sizes$chr, xlim=cbind(0,chrom_sizes$size))
  circos.genomicLabels(label_df[,c('chr','start','end')], labels=label_df$label,
    side='outside', labels.side='outside', facing='clockwise', niceFacing=TRUE,
    cex=label_cex, font=label_df$font, col=label_df$text_col, padding=0.15,
    connection_height=mm_h(2.5), labels_height=cm_h(label_height_cm),
    line_col=label_df$line_col, line_lwd=1.5, track.margin=c(0.003,0.003))
  circos.trackPlotRegion(ylim=c(0,1), track.height=0.055, bg.col='grey90', bg.border='white',
    panel.fun=function(x,y){chr=gsub('chr','',CELL_META$sector.index)
      circos.text(CELL_META$xcenter,CELL_META$ycenter,chr,facing='bending.outside',niceFacing=TRUE,cex=chr_cex,col='grey25')})
  circos.trackPlotRegion(ylim=c(0,1), track.height=0.04, bg.col=NA, bg.border=NA,
    panel.fun=function(x,y){})
  get.current.track.index()
}

draw_trimmed_arrow_link <- function(sector1, pos1, sector2, pos2, link_col,
                                    lwd=4, arr.length=0.40, arr.width=0.30, h.ratio=0.5){
  rou <- circlize:::get_most_inside_radius()
  theta1 <- circlize(pos1, 0, sector.index=sector1, track.index=0)[1, 'theta']
  theta2 <- circlize(pos2, 0, sector.index=sector2, track.index=0)[1, 'theta']
  curve <- circlize:::getQuadraticPoints(theta1, theta2, rou, rou, h=NULL, h.ratio=h.ratio, w=1)
  n_curve <- nrow(curve)
  usr <- par('usr'); cm_size <- par('pin') * 2.54
  sx <- diff(usr[1:2]) / cm_size[1]; sy <- diff(usr[3:4]) / cm_size[2]
  curve_cm <- cbind(curve[, 1] / sx, curve[, 2] / sy)
  seg_len <- sqrt(rowSums((curve_cm[-1, , drop=FALSE] - curve_cm[-n_curve, , drop=FALSE])^2))
  trim_cm <- 1.244 * arr.length
  reverse_cum <- cumsum(rev(seg_len))
  from_end <- which(reverse_cum >= trim_cm)[1]
  seg_idx <- n_curve - from_end
  prior_trim <- if(from_end == 1) 0 else reverse_cum[from_end - 1]
  fraction <- (trim_cm - prior_trim) / seg_len[seg_idx]
  base_cm <- curve_cm[seg_idx + 1, ] +
    (curve_cm[seg_idx, ] - curve_cm[seg_idx + 1, ]) * fraction
  trimmed_curve <- rbind(curve[seq_len(seg_idx), , drop=FALSE], c(base_cm[1] * sx, base_cm[2] * sy))
  lines(trimmed_curve, col=link_col, lwd=lwd, lend='butt')
  alpha <- circlize:::line_degree(curve[n_curve - 1, 1], curve[n_curve - 1, 2],
                                  curve[n_curve, 1], curve[n_curve, 2])
  shape::Arrowhead(curve[n_curve, 1], curve[n_curve, 2], alpha,
                       arr.length=arr.length, arr.width=arr.width, arr.adj=1,
                       arr.type='triangle', arr.col=link_col, lcol=link_col, arr.lwd=0.1)
}

draw_outside_cis_loop <- function(chr, pos, track_index, loop_col){
  theta <- circlize(pos, 0.5, sector.index=chr, track.index=track_index)[1, 'theta'] * pi / 180
  radial <- c(cos(theta), sin(theta)); tangent <- c(-sin(theta), cos(theta))
  loop_center <- radial * 0.84 - tangent * 0.03
  usr <- par('usr'); pin <- par('pin')
  x_radius <- 0.060
  y_radius <- x_radius * (diff(usr[3:4]) / pin[2]) / (diff(usr[1:2]) / pin[1])
  loop_angle <- seq(0.35 * pi, 2.12 * pi, length.out=120)
  loop_xy <- cbind(loop_center[1] + x_radius * cos(loop_angle),
                   loop_center[2] + y_radius * sin(loop_angle))
  n_loop <- nrow(loop_xy)
  user_per_cm <- c(diff(usr[1:2]), diff(usr[3:4])) / (pin * 2.54)
  loop_cm <- sweep(loop_xy, 2, user_per_cm, '/')
  head_length_cm <- 0.34; head_width_cm <- 0.36
  seg_len <- sqrt(rowSums((loop_cm[-1, , drop=FALSE] - loop_cm[-n_loop, , drop=FALSE])^2))
  reverse_cum <- cumsum(rev(seg_len))
  from_end <- which(reverse_cum >= head_length_cm)[1]
  seg_idx <- n_loop - from_end
  prior_trim <- if(from_end == 1) 0 else reverse_cum[from_end - 1]
  fraction <- (head_length_cm - prior_trim) / seg_len[seg_idx]
  base_cm <- loop_cm[seg_idx + 1, ] +
    (loop_cm[seg_idx, ] - loop_cm[seg_idx + 1, ]) * fraction
  base_xy <- base_cm * user_per_cm
  trimmed_loop <- rbind(loop_xy[seq_len(seg_idx), , drop=FALSE], base_xy)
  lines(trimmed_loop, col=loop_col, lwd=5.2, lend='butt')
  tip_cm <- loop_cm[n_loop, ]
  direction <- tip_cm - base_cm; direction <- direction / sqrt(sum(direction^2))
  normal <- c(-direction[2], direction[1])
  arrow_cm <- rbind(tip_cm, base_cm + normal * head_width_cm / 2,
                    base_cm - normal * head_width_cm / 2)
  arrow_xy <- sweep(arrow_cm, 2, user_per_cm, '*')
  polygon(arrow_xy, col=loop_col, border=loop_col)
  label_xy <- loop_center + c(0.095, 0.035)
  text(label_xy[1], label_xy[2], labels='cis-pQTL', col=loop_col,
       cex=1.30, font=2, adj=c(0, 0.5), xpd=NA)
}

apod <- flt %>% filter(gene_name=='APOD')
apod_v <- apod %>% group_by(cs_id) %>% slice_max(PIP,n=1,with_ties=FALSE) %>% ungroup() %>%
  transmute(cs_id,variant_ID,chr=paste0('chr',`#chr`),pos=clamp(chr,as.integer(end)),
            is_grn=variant_ID=='chr17:44352876:C:T')
apod_chr <- apod$`#chr_gene`[1]; apod_pos <- as.integer(apod$start_gene[1])
grn <- apod_v %>% filter(is_grn) %>% slice(1)
apod_labels <- tibble(
  chr=c(apod_chr,grn$chr), start=c(apod_pos,grn$pos), end=c(apod_pos+1L,grn$pos+1L),
  label=c('APOD','GRN'), text_col='black', line_col=c(c_trans_blue,highlight_blue), font=2)
apod_track <- setup_circos(apod_labels,file.path(OUT,'trans_case_circle.APOD.highlight.pdf'),1.15,1.2,8,1.1)
ord <- order(apod_v$is_grn)
for(i in ord){
  is_highlight <- apod_v$is_grn[i]
  link_col <- if(is_highlight) highlight_blue else c_trans_blue
  link_lwd <- if(is_highlight) 6.5 else 2.8
  draw_trimmed_arrow_link(apod_v$chr[i], apod_v$pos[i], apod_chr, apod_pos, link_col, lwd=link_lwd)
}
for(i in ord){
  is_grn <- apod_v$is_grn[i]
  circos.points(apod_v$pos[i],0.5,sector.index=apod_v$chr[i],track.index=apod_track,
    pch=8,cex=1.15,col=if(is_grn) highlight_blue else c_trans_blue)
}
dev.off()

fp <- gene_ref %>% filter(gene_id=='ENSG00000001036')
ft <- flt %>% filter(variant_ID %in% fuca2_cis_vars) %>% group_by(gene_name) %>%
  summarise(chr=`#chr_gene`[1], pos=as.integer(start_gene[1]), ngp=n_distinct(gpID), .groups='drop')
pal <- setNames(rep(trans_blue,nrow(ft)), ft$gene_name)
fuca_chr <- fp$chr_gene[1]; fuca_pos <- as.integer(fp$start_gene[1])
fuca_labels <- bind_rows(
  ft %>% transmute(chr,start=pos,end=pos+1L,label=gene_name,text_col='black',
                   line_col=unname(pal[gene_name]),font=1),
  tibble(chr=fuca_chr,start=fuca_pos,end=fuca_pos+1L,label='     FUCA2',
         text_col='black',line_col=trans_blue,font=2))
fuca_track <- setup_circos(fuca_labels,file.path(OUT,'trans_case_circle.FUCA2.pdf'),1.35,1.4,12.5)
for(i in seq_len(nrow(ft))){
  link_col <- unname(pal[ft$gene_name[i]])
  draw_trimmed_arrow_link(fuca_chr, fuca_pos, ft$chr[i], ft$pos[i], link_col)
}
draw_outside_cis_loop(fuca_chr, fuca_pos, fuca_track, cis_red)
circos.points(fuca_pos,0.5,sector.index=fuca_chr,track.index=fuca_track,
  pch=8,cex=2.2,col=trans_blue)
dev.off()
cat('Saved directional APOD (',nrow(apod_v),'independent trans-QTL loci ) and FUCA2 (',nrow(ft),'target genes ) to', OUT, '\n')

## 6c: APOD structure

In [ ]:
library(ggplot2)
library(cowplot)
library(magick)
library(patchwork)
ncam1_draft <- image_read_pdf("data/APOD.pdf", density = 300)  # high resolution

# Draw image-based plots (f6c, f6e)
f6f <- ggdraw() + 
  draw_image(ncam1_draft) +
  theme(plot.margin = margin(0, 0, 0, 0))

## 6d: Genotype-phenotype boxplots (GRN/APOD case)

In [ ]:
library(BEDMatrix)
library(tidyverse)
library(data.table)

anno_ref <- fread('Annotation_with_gp_coordinates_clean.txt', sep = '\t')

# Glycoprotein phenotypes (samples × glycoforms)
gp_pheno <- fread('ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/phenotype_preprocessing/unadj_glyco.sorted.bed.gz')
p_pheno <-  fread('ftp_fgc_xqtl/pQTL/ROSMAP/DLPFC/analysis_ready/phenotype_preprocessing/protein.sorted.bed.gz')
e_pheno <- fread('xqtl_data/ROSMAP/DLPFC/eQTL/pheno/DLPFC_samples_list.rnaseqc.gene_tpm.low_expression_filtered.outlier_removed.tmm.expression.remove_duplicates.bed.gz')

geno_chr17 <- BEDMatrix('FunGen_xQTL/ROSMAP/Genotype/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.17.bed')

bim_chr17 <- fread('FunGen_xQTL/ROSMAP/Genotype/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.17.bim')
fam_chr17 <- fread('FunGen_xQTL/ROSMAP/Genotype/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.17.fam')

colnames(geno_chr17) <- bim_chr17$V2 %>% gsub("_", ":", .)
rownames(geno_chr17) <- fam_chr17$V2

# prepare geno
top_var <- 'chr17:44352876:C:T'
var_ref <- str_split(top_var, ":", simplify = T) %>% .[,3]
var_alt <- str_split(top_var, ":", simplify = T) %>% .[,4]
geno0 <- paste(var_ref, var_ref, sep = '/')
geno1 <- paste(var_ref, var_alt, sep = '/')
geno2 <- paste(var_alt, var_alt, sep = '/')


geno_colors <- c(
  setNames("#1f78b4", geno0),
  setNames("#ff7f00", geno1),
  setNames("#33a02c", geno2)
)


geno_df <- geno_chr17[, top_var, drop = F] %>% as.data.frame %>% mutate(sample = rownames(.))
colnames(geno_df)[1] <- 'genotype'

In [ ]:
# prepare pheno
gpID1 = 'gp_1796'

pheno_df_gp1 <- gp_pheno %>% dplyr::filter(ID == gpID1) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>% mutate(sample = rownames(.)) %>% mutate(pheno = gpID1)
colnames(pheno_df_gp1)[1] <- 'pheno_value'

gpID2 = 'gp_1820'

pheno_df_gp2 <- gp_pheno %>% dplyr::filter(ID == gpID2) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>% mutate(sample = rownames(.)) %>% mutate(pheno = gpID2)
colnames(pheno_df_gp2)[1] <- 'pheno_value'

gene_ID = 'ENSG00000030582'
pheno_df_e <- e_pheno %>% dplyr::filter(gene_id == gene_ID) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>% mutate(sample = rownames(.)) %>% mutate(pheno = 'GRN eQTL')
colnames(pheno_df_e)[1] <- 'pheno_value'

pheno_df_p <- p_pheno %>% dplyr::filter(ID == gene_ID) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>% mutate(sample = rownames(.)) %>% mutate(pheno = 'GRN pQTL')
colnames(pheno_df_p)[1] <- 'pheno_value'

trans_ID = 'ENSG00000189058'

pheno_df_p2 <- p_pheno %>% dplyr::filter(ID == trans_ID) %>% dplyr::select(-c(1:4)) %>% t() %>% as.data.frame %>% mutate(sample = rownames(.)) %>% mutate(pheno = 'APOD pQTL')
colnames(pheno_df_p2)[1] <- 'pheno_value'

pheno_df <- rbind(pheno_df_gp1, pheno_df_gp2, pheno_df_e, pheno_df_p, pheno_df_p2)
COV_FILES <- c(
  gp = 'ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/covariate_preprocessing/unadj_glyco.sorted.covariates_unadjusted_glyco.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz',
  p  = 'data_input/protein/covariate/protein.sorted.covariates_protein.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz',
  e  = 'ftp_fgc_xqtl/eQTL/ROSMAP/DLPFC/analysis_ready/covariate_preprocessing/DLPFC_samples_list.rnaseqc.gene_tpm.low_expression_filtered.outlier_removed.tmm.expression.ROSMAP_xqtl_complete_samples_covariates_sex_death_pmi_study_transpose.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.unrelated.plink_qc.prune.pca.Marchenko_PC.gz')
read_cov <- function(f) { x <- fread(f); m <- t(as.matrix(x[, -1])); colnames(m) <- x[[1]]; m[, apply(m, 2, sd) > 0, drop = FALSE] }   # samples x covariates, zero-variance rows dropped
covs <- lapply(COV_FILES, read_cov); stopifnot(!anyNA(unlist(covs))); print(sapply(covs, dim))
adjust_geno <- function(df, cov) {
  df <- df[df$sample %in% rownames(cov) & !is.na(df$genotype), ]; df$genotype <- droplevels(df$genotype)
  X <- cbind(1, model.matrix(~ 0 + genotype, df)[, -1, drop = FALSE], scale(cov[df$sample, , drop = FALSE]))
  fit <- lm.fit(X, df$pheno_value); b <- fit$coefficients; g <- 2:nlevels(df$genotype)
  df$pheno_raw <- df$pheno_value
  df$pheno_value <- as.vector(fit$residuals + X[, g, drop = FALSE] %*% b[g]); df$pheno_value <- df$pheno_value - mean(df$pheno_value) + mean(df$pheno_raw); df
}
adjust_all <- function(d) do.call(rbind, lapply(split(d, d$pheno), function(x) adjust_geno(x, covs[[if (grepl('^gp_', x$pheno[1])) 'gp' else if (x$pheno[1] == 'GRN eQTL') 'e' else 'p']])))

In [ ]:
# plot
plot_df <- merge(pheno_df, geno_df)
glyco_final_samples <- colnames(fread('ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/covariate_preprocessing/unadj_glyco.sorted.covariates_unadjusted_glyco.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz'))[-1]
plot_df <- plot_df %>% filter(!grepl('^gp_', pheno) | sample %in% glyco_final_samples)
plot_df <- plot_df %>% mutate(genotype = ifelse(genotype == 0, geno0,
                                            ifelse(genotype == 1, geno1,
                                              ifelse(genotype == 2, geno2, genotype))))
plot_df$genotype <- factor(plot_df$genotype, levels = c(geno0, geno1, geno2))
plot_df <- adjust_all(plot_df)   # covariate-adjusted values (pheno_raw keeps the unadjusted value); samples absent from a QTL covariate file are dropped
plot_df <- plot_df %>% merge(., (anno_ref %>% dplyr::select(gpID, peptide_ID) %>% distinct), by.x = 'pheno', by.y = 'gpID', all.x = T) 
plot_df <- plot_df %>% mutate(peptide_ID = ifelse(is.na(peptide_ID), gsub(' ', '\n', pheno),gsub(' ', '\n', peptide_ID)))
plot_df$peptide_ID <- factor(plot_df$peptide_ID, levels = c('GRN\neQTL','GRN\npQTL','APOD\npQTL','APOD\nHexNAc(2)Hex(3)Fuc(1)','APOD\nHexNAc(2)Hex(2)Fuc(1)'))

med_df <- plot_df %>%
  group_by(genotype, peptide_ID) %>%
  summarise(median_y = median(pheno_value, na.rm = TRUE), .groups = "drop")

f6e <- ggplot(plot_df, aes(x = factor(genotype), y = pheno_value)) +
  geom_boxplot(width = 0.5, alpha = 0.7, outlier.shape = NA, color = "black") +
  geom_jitter(width = 0.2, size = 2, alpha = 0.6, aes(color = factor(genotype))) +
  geom_line(data = med_df,
            aes(x = as.numeric(factor(genotype)), y = median_y, group = 1),
            inherit.aes = FALSE, color = "grey50", size = 1, linetype = "dashed") +


  scale_fill_manual(values = geno_colors) +
  scale_color_manual(values = geno_colors) +
  
  labs(
    x = paste0("Genotype (", top_var, ")"),
    y = paste0("Phnotype"),
    fill = "Genotype",
    color = "Genotype"
  ) +
  
  facet_wrap(~peptide_ID, scales = 'free', nrow=1) +
  
  theme_bw(base_size = 14) +
  theme(
    strip.background = element_blank(),
    strip.text = element_text(face = "bold", size = 14),
    axis.text.x  = element_text(size = 14, face = "bold"),
    axis.text.y  = element_text(size = 14),
    axis.title   = element_text(size = 16, face = "bold"),
    legend.title = element_text(size = 14, face = "bold"),
    legend.text  = element_text(size = 12),
    legend.position  = 'none',
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    panel.border     = element_rect(size = 1.2),
    plot.title       = element_text(size = 18, face = "bold", hjust = 0.5)
  )

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)
f6e

saveRDS(list(plot_df = plot_df, geno_colors = geno_colors, top_var = top_var) , 'data/Figure_6e.rds')

## 6e: Colocboost (with hidden factors)

In [ ]:
anno_ref_path = "Annotation_with_gp_coordinates_clean.txt"
gene_ref = "resource/references/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list"

library(colocboost)
library(data.table)
library(tidyverse)
APOD <- readRDS('data_output/trans/colocboost/with_HF/APOD/transboost/transboost_AD.chr17_ENSG00000030582.tb_xqtl_AD_Bellenguez.rds')
replace_cb_id <- function(cb_res,
                          anno_ref_path = "Annotation_with_gp_coordinates_clean.txt",
                          gene_ref = "resource/references/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list") {
  # # load annotation reference
  anno_ref <- data.table::fread(anno_ref_path, sep = "\t")
  gene_ref_dt <- data.table::fread(gene_ref, sep = "\t", header = FALSE,
                                   col.names = c("chr","start","end","gene_id","gene_name"))
  
  id <- cb_res$data_info$outcome_info$outcome_names %>%
    gsub("DLPFC_Bennett_", "", .) %>%
    gsub("DLPFC_Klein_gpQTL_unadjusted_", "", .) %>%
    gsub("DeJager_", "", .)
  
  id_dt <- data.table::data.table(original_order = seq_along(id), id = id)
  
  id_dt[, gpID := stringr::str_extract(id, "^gp_\\d+")]
  
  gp_map <- anno_ref[, .(gpID, peptide_ID, ensembl_id)]
  colnames(gp_map)[3] <- 'gene_id'  
  id_dt <- merge(id_dt, gp_map, by = "gpID", all.x = TRUE)
  
  id_dt[, new_id := data.table::fifelse(!is.na(peptide_ID), peptide_ID, id)]

  id_dt[, gene_id := str_extract(new_id,'ENSG[0-9]+')]
  id_dt[, new_id := str_remove(new_id,'_ENSG[0-9]+')]
  
  id_dt <- id_dt %>% merge(.,gene_ref_dt, by = 'gene_id', all.x = T) 
  
  id_dt[, final_id := ifelse(is.na(gene_name),new_id , paste(gene_name, new_id))]
  data.table::setorder(id_dt, original_order)
  
  return(id_dt$final_id)
}

filter_APOD <- get_robust_colocalization(APOD[[1]], cos_npc_cutoff = 0.2, npc_outcome_cutoff = 0.2, pvalue_cutoff = 1e-4)

start = 5000
end = 6000

In [ ]:
source('../../2_main_figures/Figure4/data/colocboost_plot.R')
tmp <- tempfile(fileext = ".png")
png(tmp, width = 3000, height = 3000, res = 300)
colocboost_plot(filter_APOD, outcome_names = replace_cb_id(filter_APOD),
                grange = c(start:end), outcome_idx = c(42, 4, 5, 9, 41, 11, 35), 
                outcome_legend_size = 1.5, show_cos_to_uncoloc = T, plot_focal_only = T, variant_coord = TRUE)
dev.off()

region_chr = filter_APOD$data_info$variables[start] %>% str_split(., ':', simplify = T) %>% .[,1]
region_start = filter_APOD$data_info$variables[start] %>% str_split(., ':', simplify = T) %>% .[,2] %>% as.numeric
region_end = filter_APOD$data_info$variables[end] %>% str_split(., ':', simplify = T) %>% .[,2] %>% as.numeric

region = paste0(region_chr, ":",  region_start, "-", region_end)
region

In [ ]:
library(ggbio)
library(EnsDb.Hsapiens.v86)
library(ensembldb)
library(GenomicRanges)
library(ggplot2)

gene_id = 'ENSG00000030582'
gene_name = 'GRN'
region <- GRanges(region_chr %>% gsub('chr','',.), IRanges(region_start, region_end))

# longest isoform of TMEM106B
tx <- transcripts(EnsDb.Hsapiens.v86,
                  filter = GRangesFilter(region),
                  columns = c("tx_id","gene_id","gene_name","tx_strand"))

tx <- tx[mcols(tx)$gene_name == gene_name] 
tx <- tx[which.max(width(tx))]
strand_dir <- as.character(strand(tx))  # 
ex_tx <- exonsBy(EnsDb.Hsapiens.v86, by = "tx",
                 filter = TxIdFilter(tx$tx_id))[[1]]
ex_df <- as.data.frame(ex_tx)

intron_df <- data.frame(
  start = head(end(ex_tx), -1),
  end   = tail(start(ex_tx), -1),
  y     = 0
)
options(repr.plot.width=8, repr.plot.height=2)

f6g_b <- ggplot() +
  ggplot2::geom_segment(data=intron_df,
               aes(x=start, xend=end, y=0, yend=0),
               arrow = arrow(type="open", length=unit(2,"mm")),
               lineend="round", size=0.5) +
  ggplot2::geom_rect(data=ex_df,
            aes(xmin=start, xmax=end, ymin=-0.2, ymax=0.2),
            fill="steelblue") +
  theme_bw(base_size = 18) +
  xlab(gene_name) +
  ylab("") +
  theme(axis.text.y=element_blank(),
        axis.ticks.y=element_blank(),
        panel.grid=element_blank()) + ggplot2::xlim(region_start, region_end) +
  scale_x_continuous(
    limits = c(region_start, region_end),
    breaks = scales::pretty_breaks(n = 3),    
    labels = scales::label_number(scale_cut = scales::cut_short_scale())   
  )

saveRDS(list(cb_res = filter_APOD, new_id_vector = replace_cb_id(filter_APOD), intron_df = intron_df, ex_df = ex_df) , 'data/Figure_6g.rds')

## FUCA2 locus: trans-glycoQTL scope and cis-pQTL replication

In [ ]:
# trans-glycoQTL scope of the FUCA2 locus, matching Figure6a_plots.ipynb's
fuca2_lead_var <- 'chr6:143503967:G:T'
anno_ref_fp <- fread('Annotation_with_gp_coordinates_clean.txt', sep = '\t')
qtl_trans_raw <- fread('data_output/trans/fine_mapping/ROSMAP_gpQTL.exported.toploci.bed.gz')
trans_raw <- qtl_trans_raw %>% distinct() %>%
  mutate(gpID = str_extract(event_ID, 'gp_.+'), p = 2 * pnorm(-abs(z))) %>%
  filter(cs_coverage_0.95 > 0) %>%
  mutate(cs_id = paste(gene_ID, event_ID, cs_coverage_0.95, sep = '_')) %>%
  merge(anno_ref_fp, by = 'gpID', suffixes = c('_var', '_gene'))
trans_qc_fp <- trans_raw %>% filter(MAF >= 0.05, p < 5e-8) %>%
  group_by(cs_id) %>% mutate(cs_size = n_distinct(variant_ID), is_mappable = (cs_size <= 3)) %>% ungroup()
trans_lead_fp <- trans_qc_fp %>% arrange(desc(PIP)) %>% distinct(cs_id, .keep_all = TRUE)  # lead variant per CS
fuca2_trans <- trans_lead_fp %>% filter(variant_ID == fuca2_lead_var, PIP >= 0.1, is_mappable) %>%
  filter(!gpID %in% c(qc_remove_gpids, w_sole_exclude_gpids))

n_glycoproteins <- n_distinct(fuca2_trans$gene_name)
n_glycopeptiforms <- n_distinct(fuca2_trans$gpID)

pqtl <- fread('cis_association_corrected/protein/protein.sorted.bed.chr6_chr6.cis_qtl.regional.tsv.gz')
fuca2_pqtl <- pqtl %>% filter(molecular_trait_object_id == 'ENSG00000001036')

cat('FUCA2 trans locus (lead variant, PIP>=0.1, mappable CS):', n_glycopeptiforms, 'glycopeptiforms from', n_glycoproteins, 'glycoproteins\n')
cat('Target glycoproteins:', paste(sort(unique(fuca2_trans$gene_name)), collapse = ', '), '\n')
cat('FUCA2 cis-pQTL: lead variant', fuca2_pqtl$variant_id, ', nominal p =', fuca2_pqtl$p_nominal, '\n')

qc_remove_gpids <- readLines('../../2_main_figures/Figure1/data/qc_remove_gpids.txt')
w_sole_exclude_gpids <- readLines('../../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')
stopifnot(length(qc_remove_gpids) == 2, length(w_sole_exclude_gpids) == 54)
excluded_gpids <- c(qc_remove_gpids, w_sole_exclude_gpids)

# Guard the FUCA2 report object above
stopifnot(sum(unique(fuca2_trans$gpID) %in% excluded_gpids) == 0)

rds_files <- file.path('data', c('Figure_6e.rds', 'Figure_6g.rds'))
stopifnot(all(file.exists(rds_files)))
collect_strings <- function(x, depth = 0) {
  if (depth > 6) return(character(0))
  if (is.character(x) || is.factor(x)) return(as.character(x))
  if (is.data.frame(x)) return(unlist(lapply(x, function(cc) if (is.character(cc) || is.factor(cc)) as.character(cc) else NULL)))
  if (is.list(x)) return(unlist(lapply(x, collect_strings, depth = depth + 1)))
  character(0)
}
hits <- lapply(rds_files, function(f) base::intersect(unique(collect_strings(readRDS(f))), excluded_gpids))
names(hits) <- rds_files
hits <- hits[lengths(hits) > 0]
stopifnot(length(hits) == 0)
cat('No affected gpIDs (qc_remove_gpids + w_sole_exclude_gpids) found in fuca2_trans or', length(rds_files), 'Figure6_plots_preparation panel files.\n')

## APOD case: does the lead-variant effect size flip direction with vs without hidden-factor adjustment?

In [ ]:
# per-outcome effect (beta) and z-score at the GRN lead variant, from colocboost's own
APOD_noHF <- readRDS('data_output/trans/colocboost/without_HF/transboost/transboost_AD.chr17_ENSG00000030582.tb_xqtl_AD_Bellenguez.rds')[[1]]

lead_var <- 'chr17:44352876:C:T'
extract_lead_effect <- function(cb, tag) {
  idx <- which(cb$data_info$variables == lead_var)
  stopifnot(length(idx) == 1)
  data.frame(outcome_names = names(cb$data_info$coef),
             beta = sapply(cb$data_info$coef, `[`, idx),
             z    = sapply(cb$data_info$z, `[`, idx)) %>%
    setNames(c('outcome_names', paste0('beta_', tag), paste0('z_', tag)))
}

apod_hf_cmp <- merge(extract_lead_effect(APOD[[1]], 'withHF'), extract_lead_effect(APOD_noHF, 'noHF'),
                      by = 'outcome_names') %>%
  mutate(opposite_direction = sign(beta_withHF) != sign(beta_noHF),
         pct_change = 100 * (abs(beta_noHF) - abs(beta_withHF)) / abs(beta_withHF)) %>%
  arrange(outcome_names)
stopifnot(nrow(apod_hf_cmp) > 0)

n_opposite <- sum(apod_hf_cmp$opposite_direction)
cat(nrow(apod_hf_cmp), 'outcomes present in both the with-HF (n =', nrow(APOD[[1]]$data_info$outcome_info),
    ') and without-HF (n =', nrow(APOD_noHF$data_info$outcome_info), ') colocboost runs.\n')
cat(n_opposite, 'of', nrow(apod_hf_cmp), 'outcomes show an OPPOSITE effect direction with vs without HF adjustment',
    ifelse(n_opposite == 0, '-- none: direction is fully concordant with vs without HF adjustment.\n', ':\n'))
if (n_opposite > 0) print(apod_hf_cmp %>% filter(opposite_direction) %>% select(outcome_names, beta_withHF, beta_noHF))
print(apod_hf_cmp %>% mutate(across(where(is.numeric), ~round(., 4))))

cat('\neQTL and AD outcomes (not HF-adjusted phenotypes) are numerically identical between the two runs, as expected.\n')
cat('Glyco/protein outcomes (HF-adjusted phenotypes) keep the **same direction** but shrink in magnitude without HF adjustment:\n')
print(apod_hf_cmp %>% filter(grepl('gpQTL|pQTL_ENSG00000189058', outcome_names)) %>%
        transmute(outcome_names, beta_withHF = round(beta_withHF, 3), beta_noHF = round(beta_noHF, 3),
                  z_withHF = round(z_withHF, 2), z_noHF = round(z_noHF, 2), pct_change = round(pct_change, 1)))